# Gene-pair permutation subsampling

Driver notebook for `run_subsamples.py`. Two steps:

1. **Sample** — Bernoulli-subsample the transporter gene pairs in
   `<dataset>/easy_download/harreman_outputs/sample_metabolites.yaml` into a fresh
   `subsamples/run_{r}/sampled_metabolites_{j}.yaml` set (pure, local, cheap).
2. **Dispatch a SLURM job** that fits SpaceTravLR on each subsample (each surviving gene
   pair is its own `metab@{name}-{g1}_{g2}` column, both orientations summed) and writes
   the per-run analysis objects (`subsample_betas.h5ad` + `subsample_beta_means.csv`).

Setup (MAGIC/CellOracle/NicheNet) is metabolite-independent, so it is built **once per run**
and symlinked into every subsample; only `fit` re-runs per subsample. The job is resumable
(a `DONE` marker per subsample) — run it on one subsample now, resubmit to do the rest, or
call `clear_markers` to force a re-fit.

In [1]:
import sys
from pathlib import Path

# Repo root on sys.path so `metab_processing` imports work from the notebook.
_root = next(p for p in Path.cwd().resolve().parents if (p / 'setup.py').exists())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from metab_processing.metab_travlr_config import DATA_DIR
from metab_processing.SpaceTravLR.dataset_configs import get_config, dataset_paths
from metab_processing.SpaceTravLR.run_subsamples import write_subsamples, clear_markers

# --- which dataset -------------------------------------------------------------------
# Alexi UC lives under a separate data dir; its coarse annotation is the cell-type column.
dataset = '13473_HS4_UC-Slice_4'
data_dir = f'{DATA_DIR}/Alexi_UC_Spliced'
cell_type_col = '25_06_11_ICI_5K_Coarse_annotations'   # coarse annotation (CLI-overridable)
print(dataset_paths(dataset, data_dir)['selection_yaml'].parent / 'sample_metabolites.yaml')

/global/scratch/fsa/fc_wagnerlabfca/fosterangus/MetabTravLR/Data/Alexi_UC_Spliced/13473_HS4_UC-Slice_4/easy_download/harreman_outputs/sample_metabolites.yaml


## Step 1 — sample

Each of `n_permutations` files keeps every gene pair independently with probability `p`
(a metabolite left with no pairs is dropped; a wholly-empty draw is retried). This creates
the next `run_{r}` and returns `r`. Start small — Foster is running the first sample only.

In [2]:
n_permutations = 10     # how many subsamples to draw
p = 0.25                # keep-probability per gene pair (0 < p <= 1)
seed = 0               # reproducible draws (None for a fresh random run)

run = write_subsamples(dataset, n_permutations=n_permutations, p=p, data_dir=data_dir, seed=seed)
print('wrote run', run)

[17:14:54] write_subsamples: 13473_HS4_UC-Slice_4 run_2: wrote 10 files (p=0.25, seed=0) -> /global/scratch/fsa/fc_wagnerlabfca/fosterangus/MetabTravLR/Data/Alexi_UC_Spliced/13473_HS4_UC-Slice_4/easy_download/harreman_outputs/subsamples/run_2
wrote run 2


## Step 2 — dispatch the SLURM fit job

Runs `run_subsamples.py --dataset ... --run ...` on the usual GPU node. Set the knobs below;
`run=-1` uses the latest sampled run, `overwrite=True` rebuilds the shared SpaceTravLR setup.

In [3]:
from datetime import timedelta
import time

# --- knobs ---------------------------------------------------------------------------
run = -1               # which sampled run to fit (-1 = latest); or set to the printed `run`
overwrite = False      # redo the (shared) SpaceTravLR setup for this run
cpus = 8             # cores (memory scales with cores); bump for bigger datasets

cfg = get_config(dataset)
slurm = cfg['slurm']
paths = dataset_paths(dataset, data_dir)
script = Path(_root) / 'metab_processing' / 'SpaceTravLR' / 'run_subsamples.py'

cmd = ['METAB_DEBUG=1', slurm['python_path'], str(script), '--dataset', dataset,
       '--run', str(run), '--cell-type-col', cell_type_col, '--data-dir', str(data_dir)]
if overwrite:
    cmd.append('--overwrite')
command = ' '.join(cmd)

log_dir = paths['log_dir']
outlog = log_dir / f"subsamples_run{run}_{time.strftime('%Y%m%d_%H%M%S')}.log"
# sbatch_kwargs = dict(
#     account=slurm['account'], partition=slurm['partition'], qos=slurm['qos'],
#     gres=slurm['gres'], cpus_per_task=cpus, ignore_pbs=True,
#     job_name=f"MetabSub_{dataset}", output=str(outlog),
#     time=timedelta(hours=slurm['time_hours']),
# )

sbatch_kwargs = dict(
    account=slurm['account'], partition='savio4_gpu', qos='a5k_gpu4_normal',
    gres='gpu:A5000:1', cpus_per_task=cpus, ignore_pbs=True,
    job_name=f"MetabSub_{dataset}", output=str(outlog),
    time=timedelta(hours=slurm['time_hours']),
)

print('command:', command)
print('sbatch :', sbatch_kwargs)

command: METAB_DEBUG=1 /global/home/users/fosterangus/.conda/envs/spacetravlr/bin/python /global/home/users/fosterangus/Projects/MetabTravLR/SpaceTravLR/metab_processing/SpaceTravLR/run_subsamples.py --dataset 13473_HS4_UC-Slice_4 --run -1 --cell-type-col 25_06_11_ICI_5K_Coarse_annotations --data-dir /global/scratch/fsa/fc_wagnerlabfca/fosterangus/MetabTravLR/Data/Alexi_UC_Spliced
sbatch : {'account': 'fc_wagnerlabfca', 'partition': 'savio4_gpu', 'qos': 'a5k_gpu4_normal', 'gres': 'gpu:A5000:1', 'cpus_per_task': 8, 'ignore_pbs': True, 'job_name': 'MetabSub_13473_HS4_UC-Slice_4', 'output': '/global/scratch/fsa/fc_wagnerlabfca/fosterangus/MetabTravLR/Data/spacetravlr_logs/13473_HS4_UC-Slice_4/subsamples_run-1_20260913_171525.log', 'time': datetime.timedelta(days=1)}


In [4]:
# Submit. Log dir must exist before sbatch, or the job dies opening its --output file.
from simple_slurm import Slurm

log_dir.mkdir(parents=True, exist_ok=True)
job_id = Slurm(**sbatch_kwargs).sbatch(command)
print('submitted job', job_id)

Submitted batch job 38779255

submitted job 38779255


## Re-fit — clear the DONE markers

Removes the per-subsample completion markers for a run so a resubmitted job re-fits them
(SpaceTravLR still skips genes whose betadata parquet already exists — delete
`spacetravlr_subsamples/run_{r}/subsample_*/spacetravlr_output/betadata/` for a full re-fit).

In [2]:
clear_markers(dataset, run=-1, data_dir=data_dir)

[10:45:03] clear_markers: 13473_HS4_UC-Slice_1 run_2: removed 2 DONE marker(s)


2

In [12]:
import pyarrow.parquet as pq
# path = '/global/scratch/fsa/fc_wagnerlabfca/fosterangus/MetabTravLR/Data/Alexi_UC_Spliced/13473_HS4_UC-Slice_1/spacetravlr_subsamples/run_1/subsample_1/spacetravlr_output/betadata/SLC9A3_betadata.parquet'
path = '/global/scratch/fsa/fc_wagnerlabfca/fosterangus/MetabTravLR/Data/Alexi_UC_Spliced/13473_HS4_UC-Slice_1/spacetravlr_subsamples/run_1/subsample_1/spacetravlr_output/betadata/NDRG1_betadata.parquet'
print(pq.read_schema(path))

beta0: float
beta_AHR: float
beta_AHRR: float
beta_ARNT: float
beta_ATF3: float
beta_ATF4: float
beta_BATF: float
beta_BCL3: float
beta_CREB1: float
beta_CTCF: float
beta_CTCFL: float
beta_E2F1: float
beta_E2F3: float
beta_E2F4: float
beta_EGR2: float
beta_EHF: float
beta_ELK1: float
beta_ELK3: float
beta_ELK4: float
beta_EP300: float
beta_ERG: float
beta_ETS1: float
beta_ETV1: float
beta_ETV5: float
beta_ETV6: float
beta_FEV: float
beta_FLI1: float
beta_FOSL1: float
beta_FOXA1: float
beta_FOXA2: float
beta_FOXC1: float
beta_FOXC2: float
beta_FOXE1: float
beta_FOXE3: float
beta_FOXI1: float
beta_FOXJ2: float
beta_FOXL2: float
beta_FOXO1: float
beta_FOXO3: float
beta_FOXO4: float
beta_FOXP1: float
beta_FOXP2: float
beta_FOXP3: float
beta_FOXP4: float
beta_GATA1: float
beta_GATA2: float
beta_HDAC2: float
beta_HES1: float
beta_HES4: float
beta_HMGN3: float
beta_HNF4A: float
beta_IRF1: float
beta_IRF4: float
beta_JDP2: float
beta_JUN: float
beta_KLF12: float
beta_KLF14: float
beta_KLF16: f